<a href="https://colab.research.google.com/github/SavageLDN/refill-labels/blob/main/refill_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import pandas as pd
import barcode
from barcode.writer import ImageWriter

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# 1. Dataset
data = [
    {"Item": "Dish Soap Refill", "Batch_ID": "DS-2026-001", "Expiry_Date": "2027-08-18", "Volume_ml": 500},
    {"Item": "Hand Wash Refill", "Batch_ID": "HW-2026-002", "Expiry_Date": "2027-08-18", "Volume_ml": 250},
    {"Item": "Surface Cleaner", "Batch_ID": "SC-2026-003", "Expiry_Date": "2028-01-01", "Volume_ml": 1000},
    {"Item": "Laundry Detergent", "Batch_ID": "LD-2026-004", "Expiry_Date": "2027-11-30", "Volume_ml": 1500},
]
df = pd.DataFrame(data)

# 2. Function to generate a barcode image buffer in memory
def generate_barcode_buffer(code_text):
    code128 = barcode.get_barcode_class('code128')
    rv = io.BytesIO()
    barcode_instance = code128(code_text, writer=ImageWriter())
    # Generate barcode graphic without built-in text overlay to prevent double-printing
    barcode_instance.write(rv, options={"write_text": False, "quiet_zone": 1.0, "module_height": 10.0})
    rv.seek(0);
    return rv

# 3. Typography and styles
styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'LabelTitle',
    parent=styles['Normal'],
    fontSize=10,
    leading=12,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor("#111111"),
    spaceAfter=3
)
meta_style = ParagraphStyle(
    'LabelMeta',
    parent=styles['Normal'],
    fontSize=8,
    leading=10,
    fontName='Helvetica',
    textColor=colors.HexColor("#333333"),
    spaceAfter=4
)
batch_style = ParagraphStyle(
    'LabelBatch',
    parent=styles['Normal'],
    fontSize=7,
    leading=9,
    fontName='Helvetica-Bold',
    textColor=colors.HexColor("#555555"),
    alignment=1 # Centred
)

# 4. Build individual, structured label cards
label_cards = []
for _, row in df.iterrows():
    barcode_buf = generate_barcode_buffer(row['Batch_ID'])
    # Barcode image with explicit height and width
    barcode_img = RLImage(barcode_buf, width=170, height=28)

    card_elements = [
        Paragraph(row['Item'], title_style),
        Paragraph(f"Vol: {row['Volume_ml']} ml &nbsp;|&nbsp; Exp: {row['Expiry_Date']}", meta_style),
        Spacer(1, 4),
        barcode_img,
        Spacer(1, 2),
        Paragraph(f"BATCH: {row['Batch_ID']}", batch_style)
    ]

    # Place elements into a bordered label container
    card_table = Table([[card_elements]], colWidths=[240])
    card_table.setStyle(TableStyle([
        ('BOX', (0, 0), (-1, -1), 0.75, colors.HexColor("#b0b0b0")),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('LEFTPADDING', (0, 0), (-1, -1), 10),
        ('RIGHTPADDING', (0, 0), (-1, -1), 10),
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor("#ffffff")),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ]))
    label_cards.append(card_table)

# 5. Arrange into a 2-column printable A4 grid
grid_data = []
for i in range(0, len(label_cards), 2):
    row_cells = [label_cards[i]]
    if i + 1 < len(label_cards):
        row_cells.append(label_cards[i + 1])
    else:
        row_cells.append("")
    grid_data.append(row_cells)

layout_table = Table(grid_data, colWidths=[255, 255])
layout_table.setStyle(TableStyle([
    ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 14),
]))

# 6. Build PDF document
pdf_filename = "refill_labels.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=A4,
    leftMargin=20,
    rightMargin=20,
    topMargin=25,
    bottomMargin=25
)
doc.build([layout_table])
print(f"Generated clean {pdf_filename} with structured layout.")

Generated clean refill_labels.pdf with structured layout.


In [2]:
pip install reportlab

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: C:\Users\moonl\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
pip install python-barcode

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: C:\Users\moonl\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [4]:
pip install pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: C:\Users\moonl\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip
